# Christensen Dataframe Maker

Builds the analysis-ready dataframes for the christensen dilemma set:
- `christensen_event_level_df.csv`: one row per (scenario, event) -- CIK labels, per-being utility, deontic rating
- `christensen_scenario_level_df.csv`: one row per scenario -- static classification columns (evitability, personal_force, intentionality, beneficial_to), pooled utility score, %C+/%I+ proportions, and the human Likert ratings (arousal, valence, moral judgment)

Always uses **choice_1** (the harmful/instrumental action) annotated output -- christensen has annotations for both options, but only choice_1 is used throughout this analysis.

Used by `analysis_CIK.ipynb` and `analysis_utility.ipynb`.

In [1]:
from pathlib import Path
import json
import ast
import pandas as pd
import numpy as np
import sys
from IPython.display import display, HTML


In [2]:
src_path = Path().resolve() / "../../src"
sys.path.append(str(src_path))
project_root = src_path.resolve().parent
sys.path.append(str(project_root))

# christensen_utility_funcs imports src.generic_analysis_utils, so project_root must already be
# on sys.path before either import happens
import src.generic_analysis_utils as generic_analysis_utils
import christensen_utility_funcs as christensen_funcs


In [3]:
%load_ext autoreload
%autoreload 2


## Condition descriptions

- 48 first-person dilemmas, classified along 4 binary dimensions: `evitability` (evitable/inevitable), `personal_force` (personal/impersonal), `intentionality` (means/side_effect), `beneficial_to` (self/other)
- Each dilemma has two `options` annotated (choice_1, choice_2); this analysis only uses choice_1
- The annotator generates multiple outcome events per scenario, one row per event in the event-level dataframe

## Load event-level annotator dataframe (choice_1 only)

In [4]:
inputs_json = "../../scenarios_inputs/christensen/christensen_first_person_all_titled.json"
outputs_dir = "../../annotated_outputs/christensen/"

event_df = christensen_funcs.read_all_christensen_scenarios(inputs_json, outputs_dir)
# read_all_christensen_scenarios globs output files, so rows come out sorted by SID *as a
# string* (10 before 2) rather than numerically -- fix that with an explicit numeric sort.
# Every scenario_title's SIDs are already contiguous (see christensen_df_maker exploration),
# so sorting by SID alone also keeps each narrative's variants grouped together.
event_df = event_df.sort_values("SID", kind="stable").reset_index(drop=True)
display(HTML(event_df.head().to_html()))
print("number of rows:", len(event_df))
print("number of scenarios:", event_df["SID"].nunique())


,SID,option,event,C,I,K,utility,deontic,scenario_title,evitability,personal_force,intentionality,beneficial_to
0,0,1,The injured person dies,+,-,+,"{'i': 0, '1 injured person at the exit': -100}",-37.0,burning_building,evitable,personal,means,self
1,0,1,The exit blockage is broken open,+,+,+,"{'i': 60, '1 injured person at the exit': 85}",-37.0,burning_building,evitable,personal,means,self
2,0,1,The five people behind me escape the burning building,+,+,+,"{'i': 8, '1 injured person at the exit': 8}",-37.0,burning_building,evitable,personal,means,self
3,1,1,The hallway is deprived of oxygen.,+,+,+,"{'i': -80, '1 injured person crawling through the hole at the exit': -85}",12.0,burning_building,evitable,impersonal,side_effect,self
4,1,1,The fire in the hallway is extinguished.,+,+,+,"{'i': 35, '1 injured person crawling through the hole at the exit': 75}",12.0,burning_building,evitable,impersonal,side_effect,self


number of rows: 232
number of scenarios: 48


In [5]:
event_df.to_csv("christensen_event_level_df.csv", index=False)


## Load human Likert ratings (arousal, valence, moral judgment)

`human_data/christensen/human_data_formatted.csv` is keyed by a 1-indexed `id` (1-48) over the same 48 scenarios in the same order as the input JSON, so it's shifted down by 1 to line up with our 0-indexed `SID`. No ratings were collected for scenarios 21-22 (1-indexed) -- i.e. `SID` 20-21 (both `cliffhanger` variants) -- so those two rows are NaN after the merge; every downstream use of the human-rating columns has to tolerate that.

In [6]:
human_ratings_path = "../../human_data/christensen/human_data_formatted.csv"
human_ratings_df = pd.read_csv(human_ratings_path)
# file is 1-indexed (id 1-48) over the same scenarios our SID (0-47) indexes
human_ratings_df["id"] = human_ratings_df["id"] - 1
missing_ids = human_ratings_df.loc[human_ratings_df["moral_judgment_mean"].isna(), "id"].tolist()
print("SIDs with no human ratings:", missing_ids)
display(HTML(human_ratings_df.head().to_html()))


SIDs with no human ratings: [20, 21]


,id,arousal_mean,arousal_sd,valence_mean,valence_sd,moral_judgment_mean,moral_judgment_sd
0,0,5.935484,1.480706,1.935484,1.171673,3.395349,1.382536
1,1,5.629032,1.631994,2.387097,1.259128,5.279070,1.221349
2,2,6.693548,0.691606,1.177419,0.587414,2.186047,1.515881
3,3,6.612903,0.875061,1.403226,1.137436,2.325581,1.755548
4,4,5.774194,1.122342,2.467742,1.224313,5.441860,1.259295


## Build scenario-level dataframe

One row per scenario: static classification columns, %C+/%I+ proportions, the merged human ratings (NaN for SID 20-21, which have no human data), and **6 utility score variants** -- every (strategy, agg_func) combination in `christensen_utility_funcs.DEFAULT_UTILITY_VARIANTS` -- computed side by side as `utility_score_<strategy>_<agg>` columns, e.g. `utility_score_non_i_only_min`, `utility_score_simplest_mean`. `utility_score` (no suffix) is kept as an alias for `utility_score_non_i_only_min`, the original default, for anything downstream that doesn't care which variant it's using.

In [7]:
scenario_df = christensen_funcs.build_scenario_level_df(event_df, human_ratings_df=human_ratings_df)
display(HTML(scenario_df.head().to_html()))
print("number of scenarios:", len(scenario_df))
print("utility score variants:", [c for c in scenario_df.columns if c.startswith("utility_score_")])


,SID,scenario_title,evitability,personal_force,intentionality,beneficial_to,deontic,utility_score_non_i_only_min,utility_score_simplest_min,utility_score_i_only_min,utility_score,pct_C_plus,pct_I_plus,pct_I_plus_harm_only,arousal_mean,arousal_sd,valence_mean,valence_sd,moral_judgment_mean,moral_judgment_sd
0,0,burning_building,evitable,personal,means,self,-37.0,-100.0,-100.0,0.0,-100.0,100.0,66.666667,0.000000,5.935484,1.480706,1.935484,1.171673,3.395349,1.382536
1,1,burning_building,evitable,impersonal,side_effect,self,12.0,-100.0,-100.0,-80.0,-100.0,100.0,80.000000,50.000000,5.629032,1.631994,2.387097,1.259128,5.279070,1.221349
2,2,modified_crying_baby,evitable,personal,side_effect,self,-6.0,-35.0,-100.0,-100.0,-35.0,100.0,66.666667,66.666667,6.693548,0.691606,1.177419,0.587414,2.186047,1.515881
3,3,modified_crying_baby,evitable,impersonal,side_effect,self,-52.0,-100.0,-100.0,-90.0,-100.0,100.0,80.000000,80.000000,6.612903,0.875061,1.403226,1.137436,2.325581,1.755548
4,4,modified_crying_baby,evitable,personal,side_effect,self,3.0,-100.0,-100.0,-30.0,-100.0,100.0,75.000000,50.000000,5.774194,1.122342,2.467742,1.224313,5.441860,1.259295


number of scenarios: 48
utility score variants: ['utility_score_non_i_only_min', 'utility_score_simplest_min', 'utility_score_i_only_min']


In [8]:
scenario_df.to_csv("christensen_scenario_level_df.csv", index=False)
